# 🌿 Leva-TTS — Streaming Inference Server

Run the FastAPI streaming server **inside Colab** and send it requests from the same runtime.

We launch the server in the background, then hit `POST /synthesize` and the `WS /stream` endpoint.

👉 Use a **T4 GPU** runtime.

## Clone the repo + install

> We install from the **repo** (not PyPI) so the server always runs the latest
> code. We add the maintained **`coqui-tts`** fork for the XTTS engine (the old
> `TTS` won't install on Colab's Python), the repo itself, and the server deps
> (FastAPI / uvicorn / websockets). No kernel restart needed.

In [ ]:
import sys
print('Kernel Python:', sys.version.split()[0])

!git clone -q https://github.com/MohammedAly22/Leva-TTS.git
%cd /content/Leva-TTS

# Engine fork (provides the `TTS`/XTTS modules on Python 3.12)
!{sys.executable} -m pip install -q "coqui-tts[codec]==0.27.5" "transformers<5" "numpy>=2"
# Replace the original `coqpit` (Colab preinstalls it) with coqui's fork `coqpit-config`
!{sys.executable} -m pip uninstall -y -q coqpit coqpit-config
!{sys.executable} -m pip install -q --force-reinstall --no-deps coqpit-config
# The repo (editable) without re-pulling the old TTS
!{sys.executable} -m pip install -q --no-deps -e .
!{sys.executable} -m pip install -q soundfile librosa huggingface_hub rich pandas sympy "numpy>=2"
# Server deps
!{sys.executable} -m pip install -q fastapi "uvicorn[standard]" "websockets>=11" websocket-client
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
import importlib; importlib.invalidate_caches()
import leva_tts, TTS
print('✅ ready · leva_tts', leva_tts.__version__, '· TTS', TTS.__version__)

## Check the GPU

In [ ]:
# Verify a GPU runtime is active (Runtime ▸ Change runtime type ▸ T4 GPU)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "⚠️ No GPU — switch the runtime to a GPU!")

## Download the checkpoint

The server reads the checkpoint from the `LEVA_CHECKPOINT` env var, so fetch it first.

In [ ]:
import leva_tts, os
ckpt = leva_tts.download_model("/content/leva_ckpt")   # checkpoint + reference speakers
print("checkpoint at:", ckpt)
# pick any built-in reference speaker as the server default
import glob
ref = sorted(glob.glob(f"{ckpt}/reference_audios/Badr.*"))[0]
print("default speaker wav:", ref)

## Launch the server in the background

We start uvicorn as a subprocess and wait for `/health` to return OK.

In [ ]:
import os, subprocess, time, requests

env = {**os.environ, "LEVA_CHECKPOINT": ckpt, "LEVA_SPEAKER_WAV": ref,
       "LEVA_DEEPSPEED": "0"}   # deepspeed isn't installed on Colab
import sys
proc = subprocess.Popen(
    [sys.executable, "-m", "leva_tts.server.app"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# wait for startup (model load can take ~30–60s on first run)
url = "http://127.0.0.1:8000"
for _ in range(120):
    try:
        if requests.get(f"{url}/health", timeout=2).status_code == 200:
            print("✅ server is up"); break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("server did not start — check logs with: print(proc.stdout.read())")
print(requests.get(f"{url}/health").json())

## Batch request — `POST /synthesize`

Returns a WAV file; we save and play it.

In [ ]:
from IPython.display import Audio, display

r = requests.post(f"{url}/synthesize",
                  json={"text": "هَلَّق عم نجرب الـ streaming server، شو رأيك بالنتيجة؟",
                        "language": "ar", "format": "wav"})
open("server_out.wav", "wb").write(r.content)
print("status:", r.status_code, "bytes:", len(r.content))
display(Audio("server_out.wav"))

## Streaming request — `WS /stream`

Connect over WebSocket and collect PCM chunks as they arrive (measure time-to-first-audio).

In [ ]:
!pip install -q websocket-client
import websocket, json, time, numpy as np

ws = websocket.create_connection("ws://127.0.0.1:8000/stream")
ws.send(json.dumps({"text": "بِدِّي أحكيلك عن the new feature، رح تعجبك كتير هَلَّق.",
                     "language": "ar"}))

pcm, t0, ttfa = [], time.time(), None
while True:
    msg = ws.recv()
    if isinstance(msg, str):           # control / end message
        if '"event"' in msg and 'end' in msg: break
        continue
    if ttfa is None:
        ttfa = time.time() - t0
    pcm.append(np.frombuffer(msg, dtype=np.int16))
ws.close()

audio = np.concatenate(pcm).astype(np.float32) / 32768.0
print(f"time-to-first-audio: {ttfa*1000:.0f} ms · total {len(audio)/24000:.2f}s")
from IPython.display import Audio, display
display(Audio(audio, rate=24000))

## Shut the server down

In [ ]:
proc.terminate()
print("server stopped")

> The same server runs locally with:
> ```bash
> LEVA_CHECKPOINT=./checkpoints python -m leva_tts.server.app
> ```